# Re-slice the remaining real pages on the 2026-07-29 slicer

Re-cuts and re-decodes the **1,578 non-exam pages still on pre-2026-07-29 crops** and writes them
into a new root, `data/real/strips_v2`. The 203 val-side pages were already done on the laptop, so
`--skip-existing` leaves them alone.

**Why this run exists.** The slicer got four measured changes on 2026-07-29 (cap enforcement, no
overlapping crops, the staff floating in the frame, cache keys on the full windowing signature).
The app ships that slicer. Until these pages are re-cut, Round-3 *training* data is cut by
different code than production — see `docs/STATUS.md`.

⚠ **The 67 exam page images are deliberately EXCLUDED** (listed in
`data/colab/decode_pages_reslice_EXAM_EXCLUDED.txt`). The exam is frozen and read once; its 352
gold strips describe crops under `data/real/strips/`, and `exam-fix` still holds pending rows
against those crops. Producing a second set of exam crops mid-round is how the frozen exam ends up
scored on pictures its gold does not describe. Re-cutting them belongs to exam v3, deliberately.

**Three things that differ from `rung3_decode_colab.ipynb`** (the tuplet run) — do not copy cells
between the two notebooks:

| | this re-slice | the tuplet run |
|---|---|---|
| window size | **3 measures** (the default — pass no flag) | `--measures-per-strip 1` |
| output root | **`data/real/strips_v2`** | `data/real/strips` |
| model | **`round2-stage2-best`** (the live one) | `rung3-labeler` (the weak early labeler) |

⚠ **`--out` must NOT be `data/real/strips`.** Every existing manifest points into that root;
overwriting it would silently change what they refer to. Same reason the crops went to a separate
root in the first place.

⚠ **`--cache-checkpoint` must read `data/checkpoints/round2-stage2-best`.** That string is what
gets recorded in each `_decode.json`, and the laptop emitter refuses any cache whose recorded
checkpoint differs from its own `--checkpoint`. Get it wrong and all 1,578 decodes are discarded on
the way home. Cell 7 asserts this on 5 smoke pages before the full run.

## Before running

1. Upload `data/colab/tnc_reslice_colab.zip` (1.2 GB) to `MyDrive/tnc/`. **Wait for Drive to finish
   syncing** — cell 3 checks the size and will tell you if it is short.
2. Weights: if `MyDrive/tnc/round2-stage2-best/` already exists you are done. Otherwise also upload
   `data/colab/tnc_round2_ckpt.zip` (507 MB).
3. Run the cells in order. Cell 3 and cell 7 both stop the notebook on a problem rather than
   letting it surface later as a confusing error.

In [ ]:
# Which GPU did we get?
!nvidia-smi

In [ ]:
# Mount Google Drive (approve the popup).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%time
# Copy the data package Drive -> VM disk and unzip (fast local disk).
#
# TWO SEPARATE TREES, on purpose:
#   /content/tnc  = the package. Disposable — this cell deletes and re-extracts it.
#   /content/out  = the decode OUTPUT. Never touched here.
# They used to be one tree, with the output at /content/tnc/data/real/strips_v2. Re-running this
# cell after an error then silently destroyed a finished GPU run and restored the package over
# the top, leaving no trace. Output now lives outside anything this cell can delete.
#
# Written in Python, not `!`: a failing `!cp` does NOT stop the notebook, so a missing or
# still-syncing zip used to surface three cells later as a confusing "no such file".
import pathlib, shutil, subprocess

ZIP = pathlib.Path('/content/drive/MyDrive/tnc/tnc_reslice_colab.zip')
TNC = pathlib.Path('/content/tnc')
OUT = pathlib.Path('/content/out')
PAGES = 'data/colab/decode_pages_reslice.txt'

if not ZIP.exists():
    raise SystemExit(
        f'{ZIP} not found.\n'
        f'  - Upload data/colab/tnc_reslice_colab.zip to MyDrive/tnc/ (1.2 GB).\n'
        f'  - If you just uploaded it, Drive may still be syncing — wait for the upload to\n'
        f'    finish in the Drive UI, then re-run this cell.\n'
        f'  - See what is actually there:  !ls -lh /content/drive/MyDrive/tnc/')

size_gb = ZIP.stat().st_size / 1e9
print(f'found {ZIP.name}  {size_gb:.2f} GB')
if size_gb < 1.0:
    raise SystemExit(f'{ZIP} is only {size_gb:.2f} GB — expected ~1.2 GB. The upload is probably '
                     f'incomplete; re-upload and re-run.')

# Report, never delete, any work already done. --skip-existing will resume from it.
done = sum(1 for _ in (OUT / 'strips_v2').glob('*/*_decode.json')) if OUT.exists() else 0
if done:
    print(f'NOTE: {OUT}/strips_v2 already holds {done} decoded pages — kept. '
          f'The full-run cell resumes with --skip-existing.')

shutil.copy(ZIP, '/content/')
shutil.rmtree(TNC, ignore_errors=True)          # safe: the output is not in here
TNC.mkdir()
subprocess.run(['unzip', '-q', '/content/tnc_reslice_colab.zip'], cwd=TNC, check=True)
(OUT / 'strips_v2').mkdir(parents=True, exist_ok=True)

for need in (PAGES, 'scripts/rung3/decode_pages_gpu.py', 'src/vision/page_to_strips.py'):
    assert (TNC / need).exists(), f'unpacked package is missing {need}'
n = len([l for l in (TNC / PAGES).read_text().splitlines() if l.strip()])
print(f'unpacked OK — {n} pages listed, output dir {OUT}/strips_v2')
assert n > 1500, f'expected ~1578 pages, got {n} — is this an older zip?'

In [ ]:
# Dependencies (torch is preinstalled).
!pip -q install transformers opencv-python-headless

In [ ]:
%%time
# Weights -> VM disk. Prefers an unpacked copy already on Drive; else the uploaded zip.
# Prints which path it took, because a silently-wrong checkpoint is the one failure mode here
# that still produces plausible-looking output.
import pathlib, subprocess

CKPT = '/content/round2-stage2-best'
drive_ckpt = pathlib.Path('/content/drive/MyDrive/tnc/round2-stage2-best')
drive_zip = pathlib.Path('/content/drive/MyDrive/tnc/tnc_round2_ckpt.zip')

if drive_ckpt.is_dir():
    print(f'source: {drive_ckpt} (already unpacked on Drive)')
    subprocess.run(['rsync', '-a', '--exclude', 'trainer_state.pt',
                    f'{drive_ckpt}/', f'{CKPT}/'], check=True)
elif drive_zip.exists():
    print(f'source: {drive_zip}')
    subprocess.run(['unzip', '-q', '-o', str(drive_zip), '-d', '/content/ckpt'], check=True)
    subprocess.run(['rsync', '-a',
                    '/content/ckpt/data/checkpoints/round2-stage2-best/', f'{CKPT}/'], check=True)
else:
    raise SystemExit('No weights on Drive. Upload data/colab/tnc_round2_ckpt.zip (507 MB) to '
                     'MyDrive/tnc/ — or put an unpacked copy at MyDrive/tnc/round2-stage2-best/ '
                     '— then re-run this cell.')

# Verified locally 2026-07-30 that these six files are enough to load (DonutProcessor + a
# 143M-param VisionEncoderDecoder). The zip carries no preprocessor_config.json on purpose:
# this checkpoint stores the image-processor config in processor_config.json instead.
have = sorted(p.name for p in pathlib.Path(CKPT).iterdir())
print(have)
for need in ('model.safetensors', 'config.json', 'processor_config.json', 'tokenizer.json'):
    assert need in have, f'checkpoint is missing {need}'

In [ ]:
%cd /content/tnc
# SMOKE (~1 min): first 5 pages. Expect "device=cuda", staves found, 5/5 done.
#
# --out is an ABSOLUTE path outside /content/tnc, so re-running the unpack cell can never
# delete the run (it once did, costing a finished 1,506-page pass).
# NOTE: no --measures-per-strip here. The default is 3; passing 1 is the TUPLET run and would
# produce a corpus that matches nothing else.
!head -5 /content/tnc/data/colab/decode_pages_reslice.txt > /content/smoke_pages.txt
!python scripts/rung3/decode_pages_gpu.py \
    --pages /content/smoke_pages.txt \
    --checkpoint /content/round2-stage2-best \
    --cache-checkpoint data/checkpoints/round2-stage2-best \
    --out /content/out/strips_v2 \
    --batch-size 32

In [ ]:
# Check the smoke output before spending the GPU hours: the recorded checkpoint has to match what
# the laptop emitter will ask for, and the windowing signature has to be present (the laptop
# refuses caches that lack it). Absolute paths — a reconnect resets the working directory.
import json, pathlib

root = pathlib.Path('/content/out/strips_v2')
js = sorted(root.glob('*/*_decode.json'))
print(f'{len(js)} decode json(s) under {root}')
assert js, f'nothing under {root} — did the smoke cell run?'

d = json.loads(js[0].read_text())
print('checkpoint :', d.get('checkpoint'))
print('strips     :', len(d.get('strips', [])))
sig = {k: d.get(k) for k in
       ('measures_per_strip', 'window_mode', 'edge_trim', 'vplace', 'token_budget')}
print('windowing  :', sig)

assert d['checkpoint'] == 'data/checkpoints/round2-stage2-best', \
    f"cache-checkpoint is {d['checkpoint']!r} — the laptop emitter will discard every one of these"
assert d.get('measures_per_strip') == 3, 'window size is not 3 — did --measures-per-strip leak in?'
assert sig['window_mode'] is not None, 'no windowing signature — the laptop will reject these'
print('\nOK — safe to run the full pass.')

In [ ]:
%cd /content/tnc
# FULL RUN — 1,578 pages. Output goes to /content/out/strips_v2, outside the package tree, so
# nothing in this notebook can delete it. --skip-existing resumes: if the runtime drops, re-run
# cells 2-5 then this cell and it picks up where it stopped.
# Expect roughly 1.5-3 h on a T4, less on L4/A100.
# (Reference: 5.2 pages/min on an M4 CPU, so ~5 h if you ever run this locally instead.)
!python scripts/rung3/decode_pages_gpu.py \
    --pages /content/tnc/data/colab/decode_pages_reslice.txt \
    --checkpoint /content/round2-stage2-best \
    --cache-checkpoint data/checkpoints/round2-stage2-best \
    --out /content/out/strips_v2 \
    --batch-size 32 \
    --skip-existing

In [ ]:
# Did every page land? Absolute paths — a reconnect resets the working directory.
import pathlib

TNC = pathlib.Path('/content/tnc')
root = pathlib.Path('/content/out/strips_v2')
plist = TNC / 'data/colab/decode_pages_reslice.txt'

if not plist.exists():
    raise SystemExit(f'{plist} is gone — the VM was recycled. Re-run from cell 3.')
if not root.exists():
    raise SystemExit(f'{root} is missing — nothing was written. Re-run the full-run cell.')

pages = [l.strip() for l in plist.read_text().splitlines() if l.strip()]
missing = [p for p in pages
           if not (root / pathlib.Path(p).stem / f'{pathlib.Path(p).stem}_decode.json').exists()]
print(f'decoded {len(pages) - len(missing)} / {len(pages)}   missing {len(missing)}')
print(f'page dirs: {len([d for d in root.iterdir() if d.is_dir()])}   '
      f'crops: {sum(1 for _ in root.glob("*/*.png"))}')

for m in missing[:15]:
    print('   missing:', m)
if len(missing) > 15:
    print(f'   ... and {len(missing) - 15} more')

# ~4% of pages find no staves (covers, blank pages, bad scans). The laptop val-side run hit 4.6%,
# so a tail of this size is the expected outcome, not a failure.
if missing:
    pct = 100 * len(missing) / len(pages)
    print(f'\n{pct:.1f}% found no staves. Under ~6% is normal. If it is much higher, re-run the '
          f'full-run cell — --skip-existing resumes.')
else:
    print('\nAll pages decoded. Next: the pack-to-Drive cell.')

In [ ]:
%%time
# Package the crops + decode JSONs back to Drive.
#
# RUN THIS AS SOON AS THE DECODE FINISHES. Until the zip is on Drive the whole run lives only on
# an ephemeral VM disk; a recycle takes it with no warning.
import pathlib, subprocess

OUT_DIR = pathlib.Path('/content/out')          # holds strips_v2/
ZIP = pathlib.Path('/content/drive/MyDrive/tnc/reslice_strips_v2.zip')

n_json = sum(1 for _ in (OUT_DIR / 'strips_v2').glob('*/*_decode.json'))
assert n_json > 1000, f'only {n_json} decode JSONs — do not pack a partial run'
print(f'packing {n_json} decode JSONs + '
      f'{sum(1 for _ in (OUT_DIR / "strips_v2").glob("*/*.png"))} crops')

# Archived as `strips_v2/...` so it unzips into data/real/ on the laptop.
subprocess.run(['zip', '-1', '-q', '-r', str(ZIP), 'strips_v2'], cwd=OUT_DIR, check=True)
gb = ZIP.stat().st_size / 1e9
print(f'wrote {ZIP}  {gb:.2f} GB')
assert gb > 0.1, 'zip is suspiciously small — check the run before downloading'

## After the run

1. Download `MyDrive/tnc/reslice_strips_v2.zip` to the repo root.
2. Unpack it into `data/real/` — the archive holds `strips_v2/...`, so:
   ```bash
   unzip -o reslice_strips_v2.zip -d data/real/
   ```
   It merges with the 203 val-side page dirs already there.
3. Check the caches are ones the emitter will actually reuse:
   ```bash
   .venv-ml/bin/python -c "
   import json, glob, sys; sys.path.insert(0,'src/vision')
   from page_to_strips import window_cache_ok
   js = glob.glob('data/real/strips_v2/*/*_decode.json')
   d = json.load(open(js[0]))
   print(len(js), 'caches'); print(d['checkpoint'], window_cache_ok(d))"
   ```
   Expect `data/checkpoints/round2-stage2-best True`. If it prints `False`, the caches will be
   thrown away on the next emit — stop and work out why before running anything long.

⚠ **Re-emitting the training pools is a separate decision, not a formality.** It rewrites the
manifests the promoted verdicts hang off, so it wants its own `--out` and a look at what changed
before anything is promoted. Do not fold it into the unzip step.

## If a run is lost

The output lives at `/content/out/strips_v2`, deliberately outside `/content/tnc` — cell 3 deletes
and re-extracts the package tree, and when the output lived inside it, re-running that cell after
an error destroyed a finished 1,506-page pass and left no trace. Two habits that follow:

- **Pack to Drive the moment the decode finishes.** Until then the run exists only on an ephemeral
  VM disk.
- **`--skip-existing` only helps if the output survived.** It resumes from what is on disk; it
  cannot recover what was deleted.